# Praktikum 1: Rule‑Based Spam Filter — Baseline

**Learning goals for this notebook**
1. Load a small email dataset.
2. Implement a rule-based spam filter.
3. Compute baseline metrics (accuracy & confusion matrix).
4. Introduce a text-processing pipeline
5. Reflect on performance and weakneses 

## Loading libraries
In the repository, you will find the scripts to install the dependencies required for this lab under the `/scripts` folder. 
- If you are in a YourAI cluster node, make sure to run the `install_env.sh` script to download additional dependencies. 
- If you are in `google collabs`, open the terminal and load the commands in the `install_google_collabs.sh`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, ConfusionMatrixDisplay
from datasets import load_dataset, load_from_disk

## Task 1. Loading and exploring data
**Goal**: explore the SMS spam dataset and report basic dataset facts. 

**Deliverable**: Keep the code cells to perform the following operations:

1. Obtain the number of examples (rows) in the chosen split ('train').  
3. Get information about class distribution: counts for each label value (e.g., spam vs ham).  
4. Show 5 representative examples (text + label) from the dataset.
5. Remove duplicates, and keep the dataframe to only the spam and english sms message.

**Note**: Since the compute node for the jupyter notebook doesn't have access to the Internet, go you the terminal in the compute node and download the dataset with one of the provided scripts by running:

````
python scripts/download_dataset.py "dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset" data/sms_spam
````
Once downloaded, you can use `load_from_disk(path)` with the relative path to the dataset folder you downloaded. 

In [ ]:
#ds = load_dataset("dbarbedillo/SMS_Spam_Multilingual_Collection_Dataset")
ds = load_from_disk("../../data/sms_spam")
df =  ds["train"].to_pandas()

In [ ]:
# Your exploration starts here...

## Task 2. Text preprocessing pipeline
**Goal:** Implement the preprocessing pipeline by completing the missing parts.
You will evaluate and compare different preprocessing choices in Task 4. 

Use the `ntlk` library as the main reference, so that we internalize the implementation of the pipeline and its steps.
You are free to adapt the pipeline to fit the task, just add some notes of what and why.

**Deliverable:** a complete function `text_processing(text)`. 

In [ ]:
import unicodedata
import nltk
from nltk.corpus import wordnet
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk import pos_tag
import re

stemmer = SnowballStemmer("english")
wnl = WordNetLemmatizer()

def treebank_to_wordnet_pos(treebank_tag: str):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def normalize_encoding(text: str) -> str:
    # some relevant enconding operations..
    return unicodedata.normalize('NFKC', text)
    
# Let's use nltk 
def text_processing(text) :
   
    # 1) sentence segmentation on original text (we keep original for display)
    sentences = sent_tokenize(text, language='english')  

    out = []
    
    for sent in sentences:
        sent_clean  = normalize_encoding(sent)

        # 2) tokenize (on the cleaned sentence but preserve original token text for some checks)
        tokens = re.findall(r"\b\w[\w'\-]*\b", sent_clean) # to be replaced by nltk's tokenizer

        # 3) POS tagging (POS taggers expect cased text; we used cleaned but not casefolded text)
        tagged = pos_tag(tokens)            
        
        # 4) lemmatization/stemming per token (use POS for better lemmas)
        token_dicts = []
        for tok, tb_pos in tagged:
            wn_pos = treebank_to_wordnet_pos(tb_pos)
            lemma = wnl.lemmatize(tok, pos=wn_pos)
            stem = "" # stemmer... -> use the english snowball stemmer 
            token_info = {
                'text' : tok,
                'lemma' : "",    ## add missing part here
                'stem' : "",     ## add missing part here
                'casefold' : "", ## add missing part here
                'is_alpha': tok.isalpha(),
                'is_digit': tok.isdigit(),
                'is_allcaps': tok.isupper(),                
            }
            token_dicts.append(token_info)
            
        out.append({
            'sentence_raw' : sent,
            'sentence_clean' : sent_clean,
            'tokens': token_dicts
        })
    return out
    
text_processing("You have won a prize! Claim your gift now.")   

## Task 3. Rule-based spam filter
**Goal:** implement `predict_spam_rule(text) -> 1|0` using concise, interpretable rules.

Your first version should be a baseline rule system. In Task 4, you will test and improve it.

**Required basedline rules**

Implement or improve at least three rule types:

1. Keyword rule: 
Detect spam-like words such as free, win, prize.

2. Money/currency rule:
Detect simple money expressions such as $100, 100 usd, 50 eur.

3. Punctuation or style rule:
Detect suspicious writing patterns such as many ! or many ALL-CAPS words.

Use the provided code, which already highlights some that could be improved (`Improve this`). Feel free to add additional rules or logic.

**Deliverable:** the `predict_spam_rule` function plus a short comment block listing the rules and the rationale for each.

In [ ]:
import re
from typing import Tuple, Dict, List

CURRENCY_RE = re.compile(r'\$\d+|\d+\s?(usd|eur)', re.I)
URL_RE = re.compile(r'http') ## Improve this
PUNCT_RE = re.compile(r'!!!') ## Improve this
TOK_SPAM_SET = {"free", "win", "prize"} # Improve this

def predict_spam_rule(text: str) -> int:
    pred, reason = predict_spam_rule_details(text)
    return pred

def predict_spam_rule_details(text: str) -> Tuple[int, Dict]:
    """
    Returns (prediction, details) where details contains:
      - 'fired_rule': one of 'token', 'lemma', 'phrase', 'currency', 'url', 'phone',
                      'exclam', 'allcaps', 'suspicious_punct', or None
      - 'fired_detail': extra info (token/lemma/phrase/regex match)
    """
    details = {'fired_rule': None, 'fired_detail': None}
    
    # get sentence-level structures
    sentences = text_processing(text)

    for sent in sentences:        
        if CURRENCY_RE.search(sent['sentence_clean']):
            details.update({'fired_rule':'currency', 'fired_detail': CURRENCY_RE.search(sent['sentence_clean']).group(0)})
            return 1, details
            
        if URL_RE.search(sent['sentence_clean']):
            details.update({'fired_rule':'url', 'fired_detail': URL_RE.search(sent['sentence_clean']).group(0)})
            return 1, details
            
        if PUNCT_RE.search(sent['sentence_clean']):
            details.update({'fired_rule':'suspicious_punct', 'fired_detail': PUNCT_RE.search(sent['sentence_clean']).group(0)})
            return 1, details
            
        # At the token level
        tokens = sent['tokens']
        for tok in tokens:
            # check the text
            tok_text = tok.get('casefold') or tok.get('text')
            if tok_text in TOK_SPAM_SET:
                details.update({'fired_rule': 'token', 'fired_detail': tok_text})
                return 1, details
            
            # check the lemma
            lemma = tok.get('lemma') 
            if lemma and lemma.casefold() in TOK_SPAM_SET:
                details.update({'fired_rule': 'lemma', 'fired_detail': lemma})
                return 1, details            
    return 0, details

predict_spam_rule_details("You have won a prize! Claim your gift now.")

## Task 4. Apply rules to the dataset & evaluate 
**Goal:** measure how your rules perform on real data and iterate.

- Apply your rule to each example in the chosen split ('train')
- Compute and report standard metrics: **TP, FP, FN, TN**, **precision** and **recall**.

**Guided experimentation**
Try at least two of the following changes to the pipeline and observe the effects on the prediction performance:
- Compare the effect of using casefolded tokens vs raw text
- Compare using lemmas vs stems vs raw text
- Modify or improve one rules (see Task 3). You can check the errors made by your system in Task 5, and use it as the basis for your improvement.
  
Keep a log (table) of the metric values so you can show the effect of each change (an ablation table is ideal -- effect of individual).

**Deliverable:** one small results table showing baseline (no preprocessing), + preprocessing variants, and + rule changes; print the confusion counts for the final rule.


In [ ]:
y_pred = df["text"].apply(predict_spam_rule)
y_pred.value_counts()

In [ ]:
# Ground-truth labels
y_true = df["labels"] == "spam"

# Overall correctness
print('Accuracy:', accuracy_score(y_true, y_pred))

# Detailed metrics (precision, recall, F1) for each class.
# `target_names` provides human-readable labels in the SAME order scikit-learn uses:
# here 0 → 'ham', 1 → 'spam'.  If we ever swap the class order, we’d pass `labels=[1,0]`
# and provide target_names=['spam', 'ham'] in that matching order.
print(classification_report(y_true, y_pred,
                            target_names=['ham', 'spam']))


# 2×2 matrix: [[TN, FP], [FN, TP]]
cm = confusion_matrix(y_true, y_pred)

ConfusionMatrixDisplay(confusion_matrix=cm,
                       display_labels=['ham', 'spam']).plot()


## Task 5. Error analysis & short write-up
**Goal:** understand failure modes and propose concrete, explainable improvements.

- Inspect **false negatives** (spam missed) and **false positives** (ham flagged). For each selected example write 1–2 sentences:  
  - *Why* did the rule fail? (missing keyword, obfuscation, negation, ambiguity, etc.)  
  - *How* would you change the rules or preprocessing to fix it (concrete change, not vague intuition)?
- Compare evaluation **with** and **without** the preprocessing pipeline you implemented and summarize how canonicalization affected precision / recall.

**Deliverable:** a short markdown cell with at least **3 examples** (FP or FN) and your proposed fix for each, plus a 2–3 sentence summary of the overall effect of preprocessing.


In [ ]:
# Step 1: Collect all errors (where prediction != gold label)
errors = []
for i, (text, gold, pred) in enumerate(zip(df['text'], y_true, y_pred)):
    if gold != pred:
        errors.append((i, text, gold, pred))

# Step 2: Separate error types
false_positives = []  # predicted spam, but actually not spam
false_negatives = []  # predicted not spam, but actually spam

for i, text, gold, pred in errors:
    if gold == 0 and pred == 1:
        false_positives.append((i, text))
    elif gold == 1 and pred == 0:
        false_negatives.append((i, text))

# Step 3: Inspect examples
print("Example False Positives (not spam → predicted spam):")
for i, text in false_positives[:5]:
    print(f"[{i}] {text}")
    print("---")

print("\nExample False Negatives (spam → predicted not spam):")
for i, text in false_negatives[:5]:
    print(f"[{i}] {text}")
    print("---")

## General notes

### Success criteria (use for self-check)
- **Task A:** preprocessing function implemented and tested on at least two examples.  
- **Task B:** `predict_spam_rule` implemented; rules are concise and commented.  
- **Task C:** metrics computed and a short ablation table showing iterations.  
- **Task D:** at least 3 error examples analyzed with concrete fixes proposed.


**Submission:** ensure the notebook contains: (1) your `text_processing`, (2) `predict_spam_rule`, (3) evaluation table and confusion counts, and (4) the error-analysis markdown cell described above.
